# Indicadores del INE, EPHPM julio 2025 (LXXXIII)

Calcula, desde `data/raw/hogar_2025.sav`, los indicadores que publica el INE en sus cuadros de julio 2025
y los contrasta con las cifras oficiales leídas directamente de `docs/DOCS INE`.

**Premisas**
1. Las cifras principales usan las definiciones operacionales del INE: sus variables derivadas
   (`CONDACT`, `TIPINAC`, `CATEGOP`, `ACTIVIDA`, `ytraop`, `nivel`, `anosest`, `quintil`, `Pobreza`, `nbi`).
   Las lecturas literales del documento metodológico se calculan aparte, **solo para contrastar** (Paso 12).
2. No se reconstruyen decisiones internas del INE.
3. Toda contradicción o brecha queda en `docs/discrepancias_indicadores_2025.csv` (Paso 13).
4. Cada celda transforma datos, hace un `assert` o escribe un archivo.

**Salidas**
- `data/indicadores/indicadores_ine_2025.parquet` y `.csv`: tabla larga (tema, indicador, corte, categoría, valor, n_muestra).
- `docs/contraste_indicadores_ine_2025.csv`: cada cifra propia frente a la del INE.
- `docs/contraste_literal_ine_2025.csv`: definiciones literales del documento frente a las variables oficiales.
- `docs/discrepancias_indicadores_2025.csv`: discrepancias encontradas en esta corrida.

In [1]:
# ------------------------------------------------------------------ Paso 0: setup
import os, sys, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
pd.set_option("display.max_columns", 60)
print(RAIZ)

/home/claude/repo


In [2]:
# ------------------------------------------------------------------ Paso 1: fijar rutas
RAW       = Path("data/raw")
DOCS      = Path("docs")
DOCS_INE  = DOCS / "DOCS INE"
SALIDA    = Path("data/indicadores")
SALIDA.mkdir(parents=True, exist_ok=True)

ARCHIVO_SAV = RAW / "hogar_2025.sav"
assert ARCHIVO_SAV.exists(), f"Falta {ARCHIVO_SAV}"
assert DOCS_INE.exists(), f"Falta {DOCS_INE}"

## 2. Ingesta

In [3]:
# ------------------------------------------------------------------ Paso 2: ingesta
df, meta = pyreadstat.read_sav(ARCHIVO_SAV)

# Controles de la base publicada: 28,714 personas y población expandida de 10,045,495 (resumen ejecutivo)
assert df.shape == (28714, 342), df.shape
assert abs(df["FACTOR"].sum() - 10_045_495) < 1, df["FACTOR"].sum()
print("Dimensiones:", df.shape)

Dimensiones: (28714, 342)


## 3. Selección de variables

In [4]:
# ------------------------------------------------------------------ Paso 3: selección
# nombre estándar -> variable del .sav. Las variables en minúscula del .sav son derivadas por el INE.
COLS = {
    # identificación y diseño
    "hogar":            "HOGAR",
    "num_hogar":        "NUM_HOG",      # 1 = primer hogar de la vivienda
    "relacion":         "RELA_J",       # 1 = jefe(a) del hogar
    "dominio":          "domi",         # 1 DC, 2 SPS, 3 Resto urbano, 4 Rural (dominios de estimación)
    "area":             "UR",           # 1 Urbana, 2 Rural
    "factor":           "FACTOR",
    # persona
    "sexo":             "SEXO",
    "edad":             "EDAD",
    "nivel":            "nivel",        # nivel educativo del INE (agrupado)
    "anos_estudio":     "anosest",
    # condición de actividad (derivadas del INE)
    "cond_act":         "CONDACT",      # 1 Ocupado, 2 Desocupado, 3 Fuera de la FT
    "ft_potencial":     "TIPINAC",      # 1 Buscador no disponible, 2 Disponible no buscador, 3 resto
    "estudia_trabaja":  "estrab",       # 5 a 30 años
    "categoria":        "CATEGOP",
    "rama":             "RAMAOP",
    "subocup":          "ACTIVIDA",     # 2 por tiempo, 3 por ingreso
    "nsalmin":          "NSALMIN",      # 99 = no declara ingresos
    "salmin":           "SALMIN",
    "horas_op":         "TOTHRSOP",
    "ytraop":           "ytraop",       # ingreso de la ocupación principal
    "ytrab":            "ytrab",        # ingreso por trabajo total
    # hogar (derivadas del INE)
    "ypc":              "yperhg",
    "quintil":          "quintil",      # 6 = no declaran ingresos
    "pobreza":          "Pobreza",
    "nbi":              "nbi",
    "totper":           "TOTPER",       # personas que durmieron y comieron en el hogar (6 meses)
    # preguntas de la boleta (solo para el contraste literal y para los desocupados)
    "desea_trabajar":   "CA510",
    "disponible":       "CA511",
    "busco":            "CA512",
    "trabajo_antes":    "CA517",
    "tiempo_busqueda":  "CA516TIEMPO",
    "unidad_busqueda":  "CA516DSM",     # 1 días, 2 semanas, 3 meses
    "desea_mas_horas":  "CA522",
}
faltan = set(COLS.values()) - set(df.columns)
assert not faltan, faltan

base = df[list(COLS.values())].rename(columns={v: k for k, v in COLS.items()})

## 4. Etiquetas con el orden y los nombres de los cuadros del INE

In [5]:
# ------------------------------------------------------------------ Paso 4: etiquetas
# Los nombres coinciden con las filas de los cuadros, para poder cruzar sin traducir.
ETIQUETAS = {
    "dominio":   {1: "Distrito Central", 2: "San Pedro Sula", 3: "Resto urbano", 4: "Rural"},
    "area":      {1: "Urbano", 2: "Rural"},
    "sexo":      {1: "Hombre", 2: "Mujer"},
    "nivel":     {1: "Sin nivel", 2: "Basica 1-6", 3: "Basica 7-9", 4: "Media", 5: "Superior",
                  9: "No sabe, no responde"},
    "categoria": {1: "Público", 2: "Privado", 3: "Doméstico", 4: "Cuenta propia",
                  5: "Aprendiz", 6: "Trabajador familiar", 7: "Contratista dependiente"},
    "quintil":   {1: "Quintil 1", 2: "Quintil 2", 3: "Quintil 3", 4: "Quintil 4", 5: "Quintil 5",
                  6: "No declaran ingresos"},
    "pobreza":   {1: "Extrema", 2: "Relativa", 3: "No pobres"},
    "nbi":       {0: "Sin NBI", 1: "Con 1 NBI", 2: "Con 2 NBI", 3: "Con 3 y más NBI"},
    "estudia_trabaja": {1: "Estudia y trabaja", 2: "Solo trabaja", 3: "Solo estudia",
                        4: "Ni trabaja ni estudia"},
}
for col, mapa in ETIQUETAS.items():
    sin_etiqueta = set(base[col].dropna().unique()) - set(mapa)
    assert not sin_etiqueta, (col, sin_etiqueta)
    base[col + "_t"] = pd.Categorical(base[col].map(mapa), categories=list(mapa.values()), ordered=True)

# Rangos de edad de los cuadros de mercado laboral (C01 y C02)
base["rango_edad"] = pd.cut(
    base["edad"], bins=[15, 19, 25, 30, 36, 45, 60, np.inf], right=False,
    labels=["De 15 a 18 años", "De 19 a 24 años", "De 25 a 29 años", "De 30 a 35 años",
            "De 36 a 44 años", "De 45 a 59 años", "De 60 años y más"])

# Etiqueta de rama tal como viene en el .sav (21 secciones CIIU)
etq_rama = meta.variable_value_labels["RAMAOP"]
base["rama_t"] = base["rama"].map(etq_rama)

## 5. Banderas

In [6]:
# ------------------------------------------------------------------ Paso 5: banderas
b = base
b["pet"]         = b["cond_act"].notna()
b["ocupado"]     = b["cond_act"].eq(1)
b["desocupado"]  = b["cond_act"].eq(2)
b["fuera_ft"]    = b["cond_act"].eq(3)
b["ft"]          = b["ocupado"] | b["desocupado"]

# Tipos de desocupado (sección 11 del documento): cesantes y aspirantes
b["cesante"]     = b["desocupado"] & b["trabajo_antes"].eq(1)
b["aspirante"]   = b["desocupado"] & b["trabajo_antes"].eq(2)

# Fuerza de trabajo potencial (sección 17.1): variable TIPINAC
b["buscador_no_disp"] = b["ft_potencial"].eq(1)
b["disp_no_buscador"] = b["ft_potencial"].eq(2)
b["ft_potencial_b"]   = b["buscador_no_disp"] | b["disp_no_buscador"]

# Categoría ocupacional (sección 14). Asalariados = público + privado + doméstico
b["asalariado"]  = b["ocupado"] & b["categoria"].isin([1, 2, 3])

# Ingresos: el INE promedia sobre quienes declaran ingreso (NSALMIN 1 a 5)
b["declara_ingreso"] = b["ocupado"] & b["nsalmin"].between(1, 5)

# Subocupación oficial (ACTIVIDA)
b["sub_tiempo"]  = b["ocupado"] & b["subocup"].eq(2)
b["sub_ingreso"] = b["ocupado"] & b["subocup"].eq(3)

# Meses buscando trabajo. Conversión propia: 30 días y 4.33 semanas por mes
b["meses_busqueda"] = np.select(
    [b["unidad_busqueda"].eq(1), b["unidad_busqueda"].eq(2), b["unidad_busqueda"].eq(3)],
    [b["tiempo_busqueda"] / 30, b["tiempo_busqueda"] / 4.33, b["tiempo_busqueda"]], np.nan)

# Años de estudio: solo quienes tienen dato (Sin nivel y Ns/Nr vienen vacíos en anosest)
b["aep_valido"]  = b["anos_estudio"].notna()

# Trabajo infantil y juvenil: "trabaja" = tiene categoría en la ocupación principal
b["trabaja"]     = b["categoria"].notna()
b["jefe"]        = b["relacion"].eq(1)

# Reglas que deben cumplirse
assert b.loc[b["pet"], "edad"].min() >= 15
assert b.loc[~b["pet"] & b["edad"].notna(), "edad"].max() <= 14
assert (b["cesante"] | b["aspirante"]).eq(b["desocupado"]).all(), "hay desocupados sin tipo"
assert not (b["ft_potencial_b"] & ~b["fuera_ft"]).any(), "FT potencial fuera de la población fuera de la FT"
assert b.loc[b["declara_ingreso"], "ytraop"].notna().all()
assert b.loc[b["desocupado"], "meses_busqueda"].notna().all()
assert b["factor"].gt(0).all()
assert b["hogar"].nunique() == b["jefe"].sum(), "un jefe por hogar"

## 6. Funciones de estimación ponderada

In [7]:
# ------------------------------------------------------------------ Paso 6: funciones
def tot(g, m):
    return g.loc[m, "factor"].sum()

def media(g, col, m):
    x = g.loc[m, col]
    w = g.loc[m, "factor"]
    return np.average(x, weights=w) if w.sum() > 0 else np.nan

def tasa(num, den):
    return num / den * 100 if den else np.nan

def mercado_laboral(g):
    # Indicadores de los cuadros de Mercado Laboral y Problemas de Empleo
    H, M = g["sexo"].eq(1), g["sexo"].eq(2)
    pet, ft, oc, de = tot(g, g.pet), tot(g, g.ft), tot(g, g.ocupado), tot(g, g.desocupado)
    r = {
        "poblacion": g["factor"].sum(),
        "pet": pet, "pet_hombre": tot(g, g.pet & H), "pet_mujer": tot(g, g.pet & M),
        "ft": ft, "ft_hombre": tot(g, g.ft & H), "ft_mujer": tot(g, g.ft & M),
        "tp": tasa(ft, pet),
        "tp_hombre": tasa(tot(g, g.ft & H), tot(g, g.pet & H)),
        "tp_mujer": tasa(tot(g, g.ft & M), tot(g, g.pet & M)),
        "ocupados": oc, "desocupados": de, "fuera_ft": tot(g, g.fuera_ft),
        "td": tasa(de, ft), "tasa_ocupacion": tasa(oc, pet),
        "cesantes": tot(g, g.cesante), "aspirantes": tot(g, g.aspirante),
        "mbt": media(g, "meses_busqueda", g.desocupado),
        "aep_poblacion": media(g, "anos_estudio", g.aep_valido),
        "aep_pet": media(g, "anos_estudio", g.aep_valido & g.pet),
        "aep_ft": media(g, "anos_estudio", g.aep_valido & g.ft),
        "aep_ocupados": media(g, "anos_estudio", g.aep_valido & g.ocupado),
        "aep_desocupados": media(g, "anos_estudio", g.aep_valido & g.desocupado),
        "asalariados": tot(g, g.asalariado),
        "no_asalariados": oc - tot(g, g.asalariado),
        "declaran_ingreso": tot(g, g.declara_ingreso),
        "ingreso_promedio": media(g, "ytraop", g.declara_ingreso),
        "ingreso_asalariados": media(g, "ytraop", g.declara_ingreso & g.asalariado),
        "sub_tiempo": tot(g, g.sub_tiempo), "tst": tasa(tot(g, g.sub_tiempo), oc),
        "sub_ingreso": tot(g, g.sub_ingreso), "tsi": tasa(tot(g, g.sub_ingreso), oc),
        "buscador_no_disponible": tot(g, g.buscador_no_disp),
        "disponible_no_buscador": tot(g, g.disp_no_buscador),
        "ft_potencial": tot(g, g.ft_potencial_b),
        "n_pet": int(g.pet.sum()), "n_ocupados": int(g.ocupado.sum()),
        "n_desocupados": int(g.desocupado.sum()), "n_declaran": int(g.declara_ingreso.sum()),
    }
    for cod, nombre in ETIQUETAS["categoria"].items():
        clave = "cat_" + normal(nombre).replace(" ", "_")
        r[clave] = tot(g, g.ocupado & g.categoria.eq(cod))
        if cod != 6:   # el INE no promedia ingreso de trabajadores familiares
            r["ingreso_" + clave[4:]] = media(g, "ytraop", g.declara_ingreso & g.categoria.eq(cod))
    return pd.Series(r)

def normal(texto):
    # minúsculas, sin tildes, sin notas al pie ("2/") ni espacios dobles
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    t = " ".join(x for x in t.lower().split() if not x.endswith("/"))
    return t.strip()

def por_cortes(datos, funcion, cortes):
    # Devuelve una tabla larga: corte, categoria, indicador, valor
    partes = [funcion(datos).to_frame("Total Nacional").T.assign(corte="nacional")]
    for corte, col in cortes.items():
        t = datos.groupby(col, observed=True).apply(funcion, include_groups=False)
        partes.append(t.assign(corte=corte))
    ancha = pd.concat(partes).rename_axis("categoria").reset_index()
    return ancha.melt(id_vars=["corte", "categoria"], var_name="indicador", value_name="valor")

## 7. Mercado laboral, categoría ocupacional, ingresos y problemas de empleo

In [8]:
# ------------------------------------------------------------------ Paso 7: mercado laboral
CORTES_ML = {"area": "area_t", "dominio": "dominio_t", "sexo": "sexo_t",
             "nivel": "nivel_t", "rango_edad": "rango_edad"}
ml = por_cortes(base, mercado_laboral, CORTES_ML).assign(tema="mercado laboral")

# Cada corte debe sumar el total nacional en los conteos
nac = ml[ml.corte.eq("nacional")].set_index("indicador")["valor"]
for corte in ["area", "dominio", "sexo", "nivel"]:
    s = ml[ml.corte.eq(corte)].groupby("indicador")["valor"].sum()
    for ind in ["pet", "ft", "ocupados", "desocupados", "sub_tiempo", "sub_ingreso"]:
        assert abs(s[ind] - nac[ind]) < 1, (corte, ind)
assert abs(nac["ft"] - nac["ocupados"] - nac["desocupados"]) < 1
assert abs(nac["pet"] - nac["ft"] - nac["fuera_ft"]) < 1

In [9]:
# ------------------------------------------------------------------ Paso 7b: rama de actividad (ocupados)
rama = (base[base.ocupado].groupby("rama_t")["factor"].sum()
        .pipe(lambda s: pd.DataFrame({"ocupados": s, "pct_ocupados": s / s.sum() * 100}))
        .rename_axis("categoria").reset_index()
        .melt(id_vars="categoria", var_name="indicador", value_name="valor")
        .assign(corte="rama", tema="mercado laboral"))
assert abs(rama.loc[rama.indicador.eq("ocupados"), "valor"].sum() - nac["ocupados"]) < 1

## 8. Trabajo infantil (5 a 17 años) y juvenil (12 a 30 años)

In [10]:
# ------------------------------------------------------------------ Paso 8: infantil y juvenil
def condicion_trabajo(g):
    et = g["estudia_trabaja"]
    return pd.Series({
        "poblacion": g["factor"].sum(),
        "trabaja": tot(g, g.trabaja),
        "estudia_y_trabaja": tot(g, et.eq(1)),
        "solo_trabaja": tot(g, et.eq(2)),
        "solo_estudia": tot(g, et.eq(3)),
        "ni_trabaja_ni_estudia": tot(g, et.eq(4)),
        "ingreso_promedio_trabaja": media(g, "ytraop", g.trabaja & g.ytraop.notna()),
        "n_muestra": len(g),
    })

partes = []
for tema, (lo, hi) in {"trabajo infantil": (5, 17), "trabajo juvenil": (12, 30)}.items():
    sub = base[base["edad"].between(lo, hi)]
    # estrab cubre a todas las personas del rango y "trabaja" coincide con estudia y trabaja + solo trabaja
    assert sub["estudia_trabaja"].notna().all(), tema
    assert sub["trabaja"].eq(sub["estudia_trabaja"].isin([1, 2])).all(), tema
    partes.append(por_cortes(sub, condicion_trabajo, {"area": "area_t", "sexo": "sexo_t"}).assign(tema=tema))
inf_juv = pd.concat(partes, ignore_index=True)

## 9. Hogares: ingreso per cápita, quintiles, pobreza, NBI y desigualdad

In [11]:
# ------------------------------------------------------------------ Paso 9: hogares
hog = base[base.jefe].copy()      # una fila por hogar
con_ingreso = hog["pobreza"].notna()

def gini(x, w):
    x, w = np.asarray(x, float), np.asarray(w, float)
    o = np.argsort(x); x, w = x[o], w[o]
    p = np.r_[0, np.cumsum(w) / w.sum()]
    L = np.r_[0, np.cumsum(x * w) / np.sum(x * w)]
    return 1 - np.sum((L[1:] + L[:-1]) * np.diff(p))

def hogares(g):
    ci = g["pobreza"].notna()
    r = {
        "hogares": g["factor"].sum(),
        "viviendas": tot(g, g["num_hogar"].eq(1)),
        "hogares_con_ingreso": tot(g, ci),
        "ingreso_per_capita": media(g, "ypc", ci),
        "tamano_hogar": media(g, "totper", g["factor"].gt(0)),   # TH del cuadro IN C01
    }
    for cod, nom in ETIQUETAS["pobreza"].items():
        r["pct_hogares_" + normal(nom).replace(" ", "_")] = tasa(tot(g, ci & g.pobreza.eq(cod)), tot(g, ci))
    r["pct_hogares_pobres"] = tasa(tot(g, ci & g.pobreza.isin([1, 2])), tot(g, ci))
    for cod, nom in ETIQUETAS["nbi"].items():
        r["pct_hogares_" + normal(nom).replace(" ", "_")] = tasa(tot(g, ci & g.nbi.eq(cod)), tot(g, ci))
    r["pct_hogares_alguna_nbi"] = tasa(tot(g, ci & g.nbi.gt(0)), tot(g, ci))
    r["n_hogares"] = len(g)
    return pd.Series(r)

h = por_cortes(hog, hogares, {"area": "area_t", "dominio": "dominio_t"})

# Quintiles: número de hogares, tamaño e ingreso per cápita
q = hog.groupby("quintil_t", observed=True).apply(
    lambda g: pd.Series({"hogares": g["factor"].sum(),
                         "tamano_hogar": media(g, "totper", g["factor"].gt(0)),
                         "ingreso_per_capita": media(g, "ypc", g.ypc.notna()) if g.ypc.notna().any() else 0.0}),
    include_groups=False)
q = q.rename_axis("categoria").reset_index().melt(id_vars="categoria", var_name="indicador",
                                                     value_name="valor").assign(corte="quintil")

# Desigualdad y dependencia (nacional)
ypc_persona = base.groupby("hogar")["ypc"].transform("max")
mp = ypc_persona.gt(0)
mh = hog["ypc"].gt(0)
extra = pd.DataFrame({
    "indicador": ["gini_personas", "gini_hogares", "tasa_dependencia"],
    "valor": [gini(ypc_persona[mp], base.loc[mp, "factor"]),
              gini(hog.loc[mh, "ypc"], hog.loc[mh, "factor"]),
              base["factor"].sum() / nac["ocupados"]],
}).assign(corte="nacional", categoria="Total Nacional")

hg = pd.concat([h, q, extra], ignore_index=True).assign(tema="hogares")
assert abs(q.loc[q.indicador.eq("hogares"), "valor"].sum() - hog["factor"].sum()) < 1

## 10. Tabla larga de indicadores

In [12]:
# ------------------------------------------------------------------ Paso 10: unir y guardar
ind = pd.concat([ml, rama, inf_juv, hg], ignore_index=True)
ind["categoria"] = ind["categoria"].astype(str)
ind["valor"] = pd.to_numeric(ind["valor"])

# Tamaño de muestra de la celda, para marcar estimaciones poco confiables
n = (ind[ind.indicador.str.startswith("n_")]
     .pivot_table(index=["tema", "corte", "categoria"], columns="indicador", values="valor"))
ind = ind[~ind.indicador.str.startswith("n_")].merge(n, on=["tema", "corte", "categoria"], how="left")
UMBRAL = 30
ind["td_confiable"] = (ind["n_desocupados"] >= UMBRAL).astype("boolean").where(ind["tema"].eq("mercado laboral"))

ind = ind[["tema", "indicador", "corte", "categoria", "valor",
           "n_pet", "n_ocupados", "n_desocupados", "n_declaran", "n_muestra", "n_hogares", "td_confiable"]]
assert not ind.duplicated(["tema", "indicador", "corte", "categoria"]).any()

ind.to_parquet(SALIDA / "indicadores_ine_2025.parquet", index=False)
ind.to_csv(SALIDA / "indicadores_ine_2025.csv", index=False, encoding="utf-8-sig")
print(len(ind), "filas guardadas")

1289 filas guardadas


## 11. Contraste con las cifras publicadas por el INE

Las cifras oficiales se leen de los cuadros de Excel de `docs/DOCS INE`. Cada especificación indica el cuadro,
la posición del valor dentro de la fila (contando solo celdas numéricas) y el indicador propio con el que se compara.
Las cifras que solo aparecen en el resumen ejecutivo (PDF) se escriben a mano con su fuente.

In [13]:
# ------------------------------------------------------------------ Paso 11: leer cuadros del INE
ALIAS = {"ns / nr": "no sabe, no responde", "urbano 2/": "urbano"}

def leer_cuadro(archivo, hoja):
    # {etiqueta normalizada: [valores numéricos]} con la PRIMERA aparición de cada etiqueta
    crudo = pd.read_excel(DOCS_INE / archivo, sheet_name=hoja, header=None)
    filas = {}
    for _, fila in crudo.iterrows():
        celdas = [c for c in fila.tolist() if not (isinstance(c, float) and np.isnan(c))]
        if not celdas or not isinstance(celdas[0], str):
            continue
        clave = normal(celdas[0])
        clave = ALIAS.get(clave, clave)
        nums = [float(c) for c in celdas[1:] if isinstance(c, (int, float))]
        if nums and clave not in filas:
            filas[clave] = nums
    return filas

ML, PE, IN = ("Cuadros-de-Mercado-Laboral2025.xlsx", "Cuadros-de-Problemas-de-Empleo2025.xlsx",
              "Cuadros-de-Ingreso2025.xlsx")
TI, TJ = "Cuadros-de-Trabajo-Infantil2025.xlsx", "Cuadros-de-Trabajo-Juvenil2025.xlsx"

# (archivo, hoja, tema, cortes permitidos): {indicador propio: posición}
ESPEC = [
    (ML, "C01", "mercado laboral", None,
     {"pet": 0, "pet_hombre": 2, "pet_mujer": 4, "ft": 6, "ft_hombre": 8, "ft_mujer": 10,
      "tp": 12, "tp_hombre": 13, "tp_mujer": 14}),
    (ML, "C02", "mercado laboral", ["nacional", "area", "dominio", "nivel", "rango_edad", "sexo"],
     {"poblacion": 0, "aep_poblacion": 2, "aep_pet": 5, "aep_ft": 8, "ocupados": 9,
      "aep_ocupados": 11, "desocupados": 12, "aep_desocupados": 14, "td": 15, "mbt": 16}),
    (ML, "C03", "mercado laboral", None,
     {"asalariados": 2, "cat_publico": 4, "cat_privado": 6, "cat_domestico": 8, "cat_cuenta_propia": 10,
      "cat_aprendiz": 12, "cat_trabajador_familiar": 14, "cat_contratista_dependiente": 16}),
    (ML, "C04", "mercado laboral", None, {"declaran_ingreso": 2}),
    (ML, "C05", "mercado laboral", None,
     {"ingreso_promedio": 0, "ingreso_asalariados": 1, "ingreso_publico": 2, "ingreso_privado": 3,
      "ingreso_domestico": 4, "ingreso_cuenta_propia": 5, "ingreso_aprendiz": 6,
      "ingreso_contratista_dependiente": 7}),
    (PE, "C01", "mercado laboral", None, {"sub_tiempo": 7, "tst": 9, "sub_ingreso": 10, "tsi": 12}),
    (PE, "C04", "mercado laboral", ["nacional", "area", "dominio", "rango_edad"],
     {"buscador_no_disponible": 10, "disponible_no_buscador": 13}),
    (IN, "C01", "hogares", ["nacional", "area", "dominio", "quintil"],
     {"hogares": 0, "tamano_hogar": 2, "ingreso_per_capita": 3}),
    (TI, "C01", "trabajo infantil", ["nacional"],
     {"poblacion": 0, "trabaja": 2, "estudia_y_trabaja": 4, "solo_trabaja": 6, "solo_estudia": 8,
      "ni_trabaja_ni_estudia": 10}),
    (TI, "C03", "trabajo infantil", ["nacional"], {"ingreso_promedio_trabaja": 0}),
    (TJ, "C01", "trabajo juvenil", ["nacional"],
     {"poblacion": 0, "trabaja": 2, "estudia_y_trabaja": 4, "solo_trabaja": 6, "solo_estudia": 8,
      "ni_trabaja_ni_estudia": 10}),
]

claves_propias = set(map(tuple, ind[["tema", "indicador", "corte", "categoria"]].to_numpy()))
ref = []
for archivo, hoja, tema, cortes_ok, mapa in ESPEC:
    cuadro = leer_cuadro(archivo, hoja)
    propios = ind[ind.tema.eq(tema) & ind.indicador.isin(mapa)]
    if cortes_ok:
        propios = propios[propios.corte.isin(cortes_ok)]
    for (corte, cat), _ in propios.groupby(["corte", "categoria"]):
        fila = cuadro.get(normal(cat))
        if fila is None:
            continue
        for indicador, pos in mapa.items():
            if pos < len(fila) and (tema, indicador, corte, cat) in claves_propias:
                ref.append({"tema": tema, "indicador": indicador, "corte": corte, "categoria": cat,
                            "valor_ine": fila[pos], "fuente": f"{archivo} {hoja}"})
ref = pd.DataFrame(ref).drop_duplicates(["tema", "indicador", "corte", "categoria"])

# Cifras que solo están en el resumen ejecutivo (PDF)
RE = "RESUMEN-EJECUTIVO.pdf"
manual = pd.DataFrame([
    ("hogares", "pct_hogares_pobres",        "nacional", "Total Nacional", 60.1, RE + " sec. 11"),
    ("hogares", "pct_hogares_extrema",       "nacional", "Total Nacional", 38.3, RE + " sec. 11"),
    ("hogares", "pct_hogares_relativa",      "nacional", "Total Nacional", 21.9, RE + " sec. 11"),
    ("hogares", "pct_hogares_pobres",        "area",     "Urbano",         59.9, RE + " sec. 11"),
    ("hogares", "pct_hogares_pobres",        "area",     "Rural",          60.4, RE + " sec. 11"),
    ("hogares", "pct_hogares_alguna_nbi",    "nacional", "Total Nacional", 29.1, RE + " sec. 8.4"),
    ("hogares", "pct_hogares_alguna_nbi",    "area",     "Urbano",         22.4, RE + " sec. 8.4"),
    ("hogares", "pct_hogares_alguna_nbi",    "area",     "Rural",          38.5, RE + " sec. 8.4"),
    ("hogares", "hogares_con_ingreso",       "nacional", "Total Nacional", 2702184, RE + " sec. 8.4"),
    ("hogares", "viviendas",                 "nacional", "Total Nacional", 2767868, RE + " sec. 1"),
    ("hogares", "gini_personas",             "nacional", "Total Nacional", 0.495, RE + " sec. 8.3"),
    ("hogares", "gini_hogares",              "nacional", "Total Nacional", 0.495, RE + " sec. 8.3"),
    ("hogares", "tasa_dependencia",          "nacional", "Total Nacional", 2.5,   RE + " sec. 6"),
    ("mercado laboral", "pct_ocupados", "rama", "Agricultura, ganadería, silvicultura y pesca", 21.9, RE + " sec. 6"),
    ("mercado laboral", "pct_ocupados", "rama", "Manufacturera", 13.5, RE + " sec. 6"),
    ("mercado laboral", "pct_ocupados", "rama", "Construcción", 9.3, RE + " sec. 6"),
    ("mercado laboral", "pct_ocupados", "rama",
     "Comercio al por mayor y al por menor reparación de vehículos automotores y motocicletas", 23.2, RE + " sec. 6"),
], columns=["tema", "indicador", "corte", "categoria", "valor_ine", "fuente"])
ref = pd.concat([ref, manual], ignore_index=True)
assert len(ref) > 300, len(ref)

In [14]:
# ------------------------------------------------------------------ Paso 11b: comparar
cmp = ref.merge(ind[["tema", "indicador", "corte", "categoria", "valor"]],
                on=["tema", "indicador", "corte", "categoria"], how="left", validate="one_to_one")

# Tolerancia según la precisión publicada: los cuadros traen decimales; el PDF redondea a 1 decimal
es_pdf = cmp["fuente"].str.startswith(RE)
es_conteo = cmp["valor_ine"].abs() >= 1000
tol = np.select([es_conteo, es_pdf], [1.0, 0.05], 0.005)
tol = np.where(cmp["indicador"].eq("tasa_dependencia"), 0.05, tol)
tol = np.where(cmp["indicador"].isin(["gini_personas", "gini_hogares"]), 0.0005, tol)
tol = np.where(es_pdf & es_conteo, 1.0, tol)
cmp["dif"] = cmp["valor"] - cmp["valor_ine"]
cmp["dif_pct"] = np.where(cmp["valor_ine"] != 0, cmp["dif"] / cmp["valor_ine"] * 100, np.nan)
cmp["coincide"] = cmp["dif"].abs().le(tol) & cmp["valor"].notna()
cmp = cmp.sort_values(["coincide", "tema", "indicador", "corte"]).reset_index(drop=True)

# Las cifras centrales deben coincidir; si alguna falla, el notebook se detiene aquí
CENTRALES = ["pet", "ft", "ocupados", "desocupados", "td", "tp", "asalariados", "declaran_ingreso",
             "ingreso_promedio", "sub_tiempo", "sub_ingreso", "tst", "tsi", "aep_pet", "mbt",
             "buscador_no_disponible", "disponible_no_buscador", "ingreso_per_capita"]
nucleo = cmp[cmp.indicador.isin(CENTRALES) & cmp.corte.eq("nacional")]
assert nucleo["coincide"].all(), nucleo[~nucleo.coincide]

cmp.to_csv(DOCS / "contraste_indicadores_ine_2025.csv", index=False, encoding="utf-8-sig")
print(f"{cmp.coincide.sum()} de {len(cmp)} cifras coinciden")
cmp[~cmp.coincide]

931 de 947 cifras coinciden


,tema,indicador,corte,categoria,valor_ine,fuente,valor,dif,dif_pct,coincide
0,hogares,gini_hogares,nacional,Total Nacional,0.495000,RESUMEN-EJECUTIVO.pdf sec. 8.3,0.498365,0.003365,0.679756,False
1,hogares,gini_personas,nacional,Total Nacional,0.495000,RESUMEN-EJECUTIVO.pdf sec. 8.3,0.487819,-0.007181,-1.450627,False
2,hogares,tamano_hogar,quintil,No declaran ingresos,3.144317,Cuadros-de-Ingreso2025.xlsx C01,3.137844,-0.006472,-0.205843,False
3,mercado laboral,aep_desocupados,nivel,"No sabe, no responde",0.000000,Cuadros-de-Mercado-Laboral2025.xlsx C02,NaN,NaN,NaN,False
4,mercado laboral,aep_desocupados,nivel,Sin nivel,0.000000,Cuadros-de-Mercado-Laboral2025.xlsx C02,NaN,NaN,NaN,False
5,mercado laboral,aep_ft,nivel,"No sabe, no responde",0.000000,Cuadros-de-Mercado-Laboral2025.xlsx C02,NaN,NaN,NaN,False
6,mercado laboral,aep_ft,nivel,Sin nivel,0.000000,Cuadros-de-Mercado-Laboral2025.xlsx C02,NaN,NaN,NaN,False
7,mercado laboral,aep_ocupados,nivel,"No sabe, no responde",0.000000,Cuadros-de-Mercado-Laboral2025.xlsx C02,NaN,NaN,NaN,False
8,mercado laboral,aep_ocupados,nivel,Sin nivel,0.000000,Cuadros-de-Mercado-Laboral2025.xlsx C02,NaN,NaN,NaN,False
9,mercado laboral,aep_pet,nivel,"No sabe, no responde",0.000000,Cuadros-de-Mercado-Laboral2025.xlsx C02,NaN,NaN,NaN,False


## 12. Aproximaciones literales del documento metodológico (solo contraste)

Se aplican las definiciones tal como están escritas en *Metodologia-Julio-2025.pdf*, con las preguntas de la boleta.
No sustituyen a las cifras oficiales y no se usan para inferir cómo procesa el INE.

In [15]:
# ------------------------------------------------------------------ Paso 12: definiciones literales
no_ocupado = b.pet & ~b.ocupado
lit = {
    # 11. Desocupados: no ocupados que deseaban trabajar, estaban disponibles y buscaron (4 semanas)
    "desocupados_literal": no_ocupado & b.desea_trabajar.eq(1) & b.disponible.eq(1) & b.busco.eq(1),
    # 17.1.1 Buscadores no disponibles: buscaron y no están disponibles, pero lo estarán pronto
    "buscador_no_disp_literal": no_ocupado & b.busco.eq(1) & b.disponible.isin([2, 3]),
    # 17.1.2 Disponibles no buscadores: desean trabajar, disponibles, no buscaron
    "disp_no_buscador_literal": no_ocupado & b.desea_trabajar.eq(1) & b.disponible.eq(1) & b.busco.eq(2),
    # 10.1 Subocupación visible: < 40 h y desea trabajar más (horas: solo ocupación principal)
    "sub_tiempo_literal": b.ocupado & b.horas_op.lt(40) & b.desea_mas_horas.eq(1),
    # 10.2 Subocupación invisible: >= 40 h e ingreso mensual (principal + secundaria) < salario mínimo
    "sub_ingreso_literal": b.ocupado & b.horas_op.ge(40) & b.ytrab.lt(b.salmin),
}
oficial = {"desocupados_literal": b.desocupado, "buscador_no_disp_literal": b.buscador_no_disp,
           "disp_no_buscador_literal": b.disp_no_buscador, "sub_tiempo_literal": b.sub_tiempo,
           "sub_ingreso_literal": b.sub_ingreso}
literal = pd.DataFrame([{
    "indicador": k,
    "literal": tot(b, m),
    "oficial": tot(b, oficial[k]),
    "ambos": tot(b, m & oficial[k]),
    "solo_literal": tot(b, m & ~oficial[k]),
    "solo_oficial": tot(b, ~m & oficial[k]),
} for k, m in lit.items()])
oc_tot = nac["ocupados"]
literal["tasa_literal"] = np.where(literal.indicador.str.startswith("sub_"), literal.literal / oc_tot * 100, np.nan)
literal["tasa_oficial"] = np.where(literal.indicador.str.startswith("sub_"), literal.oficial / oc_tot * 100, np.nan)
assert (literal["ambos"] + literal["solo_literal"] - literal["literal"]).abs().max() < 1
literal.to_csv(DOCS / "contraste_literal_ine_2025.csv", index=False, encoding="utf-8-sig")
literal.round(1)

,indicador,literal,oficial,ambos,solo_literal,solo_oficial,tasa_literal,tasa_oficial
0,desocupados_literal,197159.3,211014.6,197159.3,0.0,13855.3,NaN,NaN
1,buscador_no_disp_literal,288.1,26832.5,0.0,288.1,26832.5,NaN,NaN
2,disp_no_buscador_literal,213957.7,119999.3,105357.4,108600.3,14641.9,NaN,NaN
3,sub_tiempo_literal,277683.7,327692.4,277683.7,0.0,50008.7,6.8,8.0
4,sub_ingreso_literal,1611084.1,1359616.6,1350934.7,260149.4,8681.9,39.5,33.4


## 13. Registro de discrepancias de esta corrida

In [16]:
# ------------------------------------------------------------------ Paso 13: discrepancias
L = literal.set_index("indicador")
def f0(x): return f"{x:,.0f}"
def f1(x): return f"{x:,.1f}"
def v(tema, ind_, corte="nacional", cat="Total Nacional"):
    return ind.set_index(["tema", "indicador", "corte", "categoria"]).loc[(tema, ind_, corte, cat), "valor"]

ft15 = tot(b, b.ft & b.edad.eq(15))
anos_raros = int((b.anos_estudio > 30).sum())
alguna_nbi_todos = tasa(tot(hog, hog.nbi.gt(0)), hog.factor.sum())
ypc_con_ceros = np.average(hog["ypc"].fillna(0), weights=hog["factor"])

DISC = [
 ("P01", "Desocupados: total en el texto", "Resumen sec. 7: 'del total de 250,668 desocupados'",
  f"Cuadros y base: {f0(nac['desocupados'])}", f"{f0(250668 - nac['desocupados'])} de diferencia",
  "Media", "abierta", "Usar el cuadro ML C02; el texto no cuadra con ninguna cifra publicada"),
 ("P02", "TSI: valor en el texto", "Resumen sec. 7: TSI nacional 34.4%",
  f"Cuadro PE C01 y base: {f1(nac['tsi'])}%", "1 punto", "Baja", "abierta", "Usar el cuadro"),
 ("P03", "Quintiles: texto invertido", "Resumen sec. 10: primer quintil L.12,967 y último L.710",
  f"Cuadro IN C01 y base: Q1 {f0(v('hogares','ingreso_per_capita','quintil','Quintil 1'))}, "
  f"Q5 {f0(v('hogares','ingreso_per_capita','quintil','Quintil 5'))}", "Errata", "Baja", "abierta",
  "Usar el cuadro"),
 ("P04", "Edad de la fuerza de trabajo", "Doc. sec. 9: 'mayores de 15 años'; sec. 8: PET 15 años y más",
  f"La FT incluye personas de 15 años: {f0(ft15)}", "Contradicción del documento", "Baja", "abierta",
  "Seguir a la PET (15 y más), que es lo que hace CONDACT"),
 ("P05", "Iniciadores", "Doc. sec. 11: forman parte de los desocupados y a la vez 'no estaban ni ocupados ni desocupados'",
  "Cesantes + aspirantes = 100% de los desocupados; no hay variable de iniciadores",
  f"Cesantes {f0(nac['cesantes'])}, aspirantes {f0(nac['aspirantes'])}", "Media", "abierta",
  "No se estiman iniciadores"),
 ("P06", "Etiquetas de TIPINAC", "Cuadro PE C04: 'Buscador no disponible' y 'Disponible no buscador'",
  "El .sav las etiqueta 'Potencialmente activos' y 'Desalentados'; el doc dice que 'potencialmente activos' "
  "era el nombre antiguo de los iniciadores", "Las cifras coinciden con el cuadro", "Baja", "cerrada",
  "Usar los nombres del cuadro"),
 ("P07", "Desocupados: definición literal", "Doc. sec. 11: no ocupado, desea, disponible y buscó",
  f"Literal {f0(L.loc['desocupados_literal','literal'])} frente a oficial {f0(nac['desocupados'])}",
  f"Solo oficial: {f0(L.loc['desocupados_literal','solo_oficial'])}; solo literal: "
  f"{f0(L.loc['desocupados_literal','solo_literal'])}", "Media", "abierta",
  "Cifra principal = CONDACT; la literal solo como contraste"),
 ("P08", "Fuerza de trabajo potencial: definición literal", "Doc. sec. 17.1.1 y 17.1.2",
  f"BND literal {f0(L.loc['buscador_no_disp_literal','literal'])} (oficial "
  f"{f0(nac['buscador_no_disponible'])}); DNB literal {f0(L.loc['disp_no_buscador_literal','literal'])} "
  f"(oficial {f0(nac['disponible_no_buscador'])})", "El DNB literal casi duplica al oficial", "Media",
  "abierta", "Cifra principal = TIPINAC; la literal solo como contraste"),
 ("P09", "Subocupación: definición literal", "Doc. sec. 10.1 y 10.2: umbral de 40 h",
  f"Literal {f1(L.loc['sub_tiempo_literal','tasa_literal'])}% / {f1(L.loc['sub_ingreso_literal','tasa_literal'])}% "
  f"frente a oficial {f1(nac['tst'])}% / {f1(nac['tsi'])}%", "Ver D01 y D02", "Alta", "abierta",
  "Cifra principal = ACTIVIDA"),
 ("P10", "Años de estudio promedio (AEP)", "Cuadros ML C02 y PE C01 muestran AEP = 0 para 'Sin nivel'",
  f"El AEP total coincide solo si se excluye 'Sin nivel' (anosest vacío); incluye {anos_raros} casos con "
  "anosest entre 108 y 111", "AEP total sobrestimado frente a contar 'Sin nivel' como 0", "Media", "abierta",
  "Reproducir al INE; documentar"),
 ("P11", "Coeficiente de Gini", "Resumen sec. 8.3: 0.495; no define la unidad ni el ingreso",
  f"Personas: {v('hogares','gini_personas'):.3f}; hogares: {v('hogares','gini_hogares'):.3f}",
  "Ninguna variante coincide", "Baja", "abierta", "Reportar ambas con advertencia"),
 ("P12", "NBI: base de hogares", "Resumen sec. 8.4: 29.1% de 2,702,184 hogares",
  f"2,702,184 = hogares con ingreso declarado; con los {f0(hog.factor.sum())} hogares: "
  f"{f1(alguna_nbi_todos)}%", "0.2 puntos", "Baja", "abierta", "Reproducir al INE; documentar la base"),
 ("P13", "Ingreso per cápita: base", "Resumen sec. 10: 'con base en un total de 2,780,759 hogares'",
  f"L.4,864 sale de los hogares con ingreso declarado; contando todos con 0: {f0(ypc_con_ceros)}",
  "Texto impreciso", "Baja", "abierta", "Usar hogares con ingreso"),
 ("P14", "Cuadro ML C02: filas de rama", "Filas Primaria, Secundaria y Terciaria",
  "Los valores están desplazados (la fila 'Primaria' repite los totales de mujeres)",
  "Cuadro inutilizable por rama", "Baja", "abierta", "Usar la distribución por rama de la base (Paso 7b)"),
 ("P15", "Cuadro TJ C01: bloque urbano", "Bloque 'Urbano'",
  "La fila 'Sexo' contiene los totales urbanos; el bloque está desplazado", "Formato", "Baja", "abierta",
  "Contrastar solo el total nacional"),
 ("P16", "Pobreza: redacción", "Resumen sec. 11: 'la pobreza extrema... 59.9% urbano y 60.4% rural'",
  f"59.9/60.4 son pobreza total; extrema urbana/rural: {f1(v('hogares','pct_hogares_extrema','area','Urbano'))}/"
  f"{f1(v('hogares','pct_hogares_extrema','area','Rural'))}. El texto dice 'integrados 33.2%' y el gráfico 30.4%",
  "Errata", "Baja", "abierta", "Usar las cifras de las tablas"),
 ("P17", "No respuesta de la muestra", "Doc. cuadro 3: 28.0% de no respuesta (DC 43.9%, SPS 41.3%)",
  "El factor ya trae el ajuste A1; la precisión urbana es menor", "Afecta la confiabilidad por dominio",
  "Media", "abierta", "Marcar celdas con td_confiable"),
 ("P19", "Tamaño del hogar", "Cuadro IN C01: TH nacional 3.64; resumen: 3.6 personas por hogar",
  f"TH = promedio de TOTPER ({v('hogares','tamano_hogar'):.3f}); población / hogares = "
  f"{base['factor'].sum() / hog['factor'].sum():.3f}", "TOTPER no cuadra con las personas de la base",
  "Baja", "abierta", "Usar TOTPER para reproducir el cuadro; documentar"),
 ("P20", "Celdas vacías como cero", "Cuadros ML C02 y C05 escriben 0 cuando no hay casos (AEP de Sin nivel, aprendices)",
  "La base no tiene casos en esas celdas", "Un 0 publicado puede no ser un valor medido", "Baja", "abierta",
  "Tratar los 0 del INE en esas celdas como vacíos"),
 ("P18", "Ingreso promedio de los ocupados", "Cuadro ML C05: ocupación principal, quienes declaran ingreso",
  f"ytraop sobre NSALMIN 1-5 reproduce {f1(nac['ingreso_promedio'])}; ytrab da "
  f"{f1(media(b, 'ytrab', b.declara_ingreso))}", "Sin brecha con ytraop", "Alta", "cerrada",
  "Usar ytraop para ingresos de la ocupación principal (relacionada con D22)"),
]
disc = pd.DataFrame(DISC, columns=["id", "tema", "ine_dice", "base_muestra", "magnitud",
                                   "prioridad", "estado", "decision"])
assert disc["id"].is_unique
disc.to_csv(DOCS / "discrepancias_indicadores_2025.csv", index=False, encoding="utf-8-sig")
print(len(disc), "discrepancias registradas")

20 discrepancias registradas
